# 🐍 **The Beginner's Introduction to Asynchronous Programming in Python**

Welcome to your guide on **Asynchronous Programming** in Python! Asynchronous code allows you to handle multiple tasks concurrently without blocking your program, making it essential for high-performance applications like web servers, scrapers, and API integrations.

### 🎯 **What You'll Learn**
*   How to define and execute **coroutines**.
*   The magic of the `await` keyword.
*   Difference between **Concurrency vs. Parallelism**.

---
## 🛠️ Section 1: The Core Keywords
Before we dive into complex logic, we need to understand the two pillars of async Python: `async def` and `await`.

* ### `async def` - define a coroutine





We will first define an async function for demonstration.

In [1]:
async def fetch_data():
  return "data"

Now, we may ask what happens when we call it.

In [2]:
result = fetch_data()
print(result)

<coroutine object fetch_data at 0x7c7fc26001a0>


You get `<coroutine object fetch_data at 0x7c7fc26001a0>`.

If you were expecting the string output "data", then you have been bamboozled.

* ### `await` - pause execution


Inside an async function, `await` pauses execution until "something" finishes.

In [4]:
import asyncio

async def say_hello():
  await asyncio.sleep(1) # non-blocking delay
  print("Hello")

**Key Point:** `await` only works inside `async def`.

**TRY THIS!**

Replace `async def` with `def`.

**OBSERVATION :** You will notice a syntax error occurs.

**NOTE:** *We use the `asyncio` library with the `sleep` attribute to demonstrate `await`.*

* ### event loops


Python uses an event loop via `async` to manage all async tasks.

In [9]:
# asyncio.run(say_hello())

await say_hello()

Hello


**NOTE:** `asyncio.run(say_hello())` results in `RuntimeError: asyncio.run() cannot be called from a running event loop` due to trying to call `asyncio.run()` in a Colab environment where an event loop is already active.

**FIX:** For environments where an event loop is already active, we may just use `await say_hello()`.

* ### sequential v/s async

❌ Blocking version

In [16]:
import time

def task(name):
  print(f"Start {name}")
  time.sleep(1)
  print(f"End {name}")

tasks = ["A", "B"]

start = time.time()

for task_name in tasks:
    task(task_name)

end = time.time()
print(f"Total time taken: {end - start}")

Start A
End A
Start B
End B
Total time taken: 2.000789165496826


✅ Async version

In [15]:
async def task(name):
  print(f"Start {name}")
  await asyncio.sleep(1)
  print(f"End {name}")

async def main():
  start = time.time()
  await  asyncio.gather(
      task("A"),
      task("B")
  )
  end = time.time()
  print(f"Total time taken: {end - start}")

# asyncio.run(main())
await main() # for Colab-like environmemnts

Start A
Start B
End A
End B
Total time taken: 1.0019454956054688


### So what happened?

The async version was faster because `task A` and `task B` happened **concurrently**.

* **Concurrency** is about managing multiple tasks at once.

* **Parallelism** is about executing multiple tasks at the same time.

---
## 🏗️ Section 2: Concurrency vs. Parallelism
Understanding the difference between these two is vital for knowing when to use `asyncio`.

### **🔄 Concurrency: taking turns efficiently**

In a concurrent system, tasks do not all run simultaneously. They take turns instead.

**Only one thing is happening at any exact moment, but progress is made on multiple tasks.**

That is what `asyncio` does:

* One thread
* One event loop
* Tasks pause (await) and let others run

So the async example worked faster because:

**While Task A was “sleeping,” Task B got a turn.**

### **⚡ Parallelism: truly simultaneous execution**

Parallelism means tasks are literally running at the same time.

**Work is happening simultaneously on different CPU cores.**

In Python, this usually involves:

* `multiprocessing`
* Native threads (with caveats like the GIL)

---

## ❓ Section 3: Deep Dive Q&A
Before we build more, let's clear up some common confusion about blocking vs. non-blocking code.

### Q1: Why does `time.sleep()` do different to `await asyncio.sleep()`?

### ANSWER:

`time.sleep(1)` tells python to do **nothing** for 1 second.

**It blocks the entire thread.**

In our example, when it is sleeping in task A, it cannot start task B.

So timeline is:

```
Start A
(wait 1s doing nothing)
End A
Start B
(wait 1s doing nothing)
End B
```

`await asyncio.sleep(1)` tells event loop: I am waiting for 1 second. You can run something else now.

### Q2: In the async scenario where our function does actual work for 1.2 seconds, followed by the `await asyncio.sleep(1)`, what is the timeline for tasks A and B?

### ANSWER:

This is the timeline:

```
t = 0.0     Task A starts
            Does 1.2s CPU work
            ⚠️ This blocks everything

0.0 → 1.2   A CPU work, nothing else runs

t = 1.2     Task A hits: `await asyncio.sleep(1)`
            Now A is “sleeping” until t = 2.2
            Event loop switches to Task B

1.2 → 2.4   B CPU work, ⚠️ This blocks the event loop again

🚨 Key moment
Task A’s sleep was supposed to finish at: t = 2.2
But the event loop is busy running Task B CPU work until: t = 2.4
So what happens?
Task A cannot resume at 2.2 — it is delayed until t = 2.4

t = 2.4     Task B finishes CPU work
            Now event loop is free again
            Task A sleep is already overdue → resumes immediately → End A
            Task B hits await sleep(1) → scheduled until t = 3.4

2.4 → 3.4   B sleeping

t = 3.4     Task B finishes → End B
```

---
